# Recommendation detection: compare saved models

Load the three trained model artifacts and evaluate them on the same reserved test rows. Show the training CV macro-F1 alongside test metrics; the test set was already examined during development, so its scores are exploratory. Review the results before manually filling in the final save cell.


In [1]:
from pathlib import Path
from hashlib import sha256
import json
import pandas as pd

LABELS = {"not_recommended": 0, "recommended": 1}
LABEL_NAMES = {value: key for key, value in LABELS.items()}
PROJECT_ROOT = next(
    (root for root in (Path.cwd(), *Path.cwd().parents)
     if (root / "data" / "train.parquet").is_file()
     and (root / "data" / "test.parquet").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run prepare_data.ipynb to create data/train.parquet and data/test.parquet.")
train_path = PROJECT_ROOT / "data" / "train.parquet"
test_path = PROJECT_ROOT / "data" / "test.parquet"
model_dir = PROJECT_ROOT / "notebooks" / "recommendation detection" / "output models"
train = pd.read_parquet(train_path, engine="pyarrow")
test = pd.read_parquet(test_path, engine="pyarrow").reset_index(drop=True)
required = {"source_row", "body", "processed_body", "recommendation_status"}
if not required.union({"fold"}).issubset(train.columns) or not required.issubset(test.columns):
    raise ValueError("Prepared data columns are missing.")
assert set(train["source_row"]).isdisjoint(test["source_row"])
assert set(train["processed_body"]).isdisjoint(test["processed_body"])
assert set(test["recommendation_status"]) == set(LABELS.values())
assert not test[list(required)].isna().any().any()
y_test = test["recommendation_status"]

data_hashes = {
    "train_sha256": sha256(train_path.read_bytes()).hexdigest(),
    "test_sha256": sha256(test_path.read_bytes()).hexdigest(),
}
artifact_paths = {
    "TF-IDF": (model_dir / "tfidf_pipeline.joblib", model_dir / "tfidf_metrics.json"),
    "fastText": (model_dir / "fasttext.bin", model_dir / "fasttext_metrics.json"),
    "Frozen ParsBERT": (model_dir / "frozen_parsbert", model_dir / "frozen_parsbert" / "metrics.json"),
}
metrics = {}
for approach, (artifact, metadata_path) in artifact_paths.items():
    if not artifact.exists() or not metadata_path.is_file():
        raise FileNotFoundError(f"Missing {approach} artifact or metrics. Run its training notebook.")
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata["approach"] != approach or any(metadata.get(k) != v for k, v in data_hashes.items()):
        raise ValueError(f"{approach} was trained with a different train/test file. Rerun its notebook.")
    metrics[approach] = metadata
print("Shared test rows:", len(test))


Shared test rows: 1999


## Prepare inputs and load saved models


In [2]:
import string
import hazm

normalizer = hazm.Normalizer()


def remove_chars(text, chars):
    for char in chars:
        text = text.replace(char, " ")
    return text


def preprocess_light(text):
    if not isinstance(text, str):
        return None
    text = remove_chars(text.lower(), string.punctuation + "؟،")
    text = normalizer.normalize(text).replace("آ", "ا")
    text = remove_chars(text, "۰۱۲۳۴۵۶۷۸۹")
    return " ".join(text.split()) or None

X_test_light = test["body"].apply(preprocess_light)
if X_test_light.isna().any():
    raise ValueError("An empty test comment appeared after light normalization.")


In [3]:
import joblib
import fasttext
import torch
import numpy as np
from transformers import AutoModel, AutoTokenizer

tfidf_model = joblib.load(artifact_paths["TF-IDF"][0])
fasttext_model = fasttext.load_model(str(artifact_paths["fastText"][0]))
bert_dir = artifact_paths["Frozen ParsBERT"][0]
bert_manifest = json.loads((bert_dir / "manifest.json").read_text(encoding="utf-8"))
if bert_manifest["text_preprocessing"] != "preprocess_light" or bert_manifest["pooling"] != "attention_masked_mean":
    raise ValueError("Unsupported ParsBERT preprocessing or pooling settings.")
bert_tokenizer = AutoTokenizer.from_pretrained(bert_dir / "tokenizer", local_files_only=True)
bert_encoder = AutoModel.from_pretrained(bert_dir / "encoder", local_files_only=True)
bert_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_encoder = bert_encoder.to(bert_device).eval()
bert_classifier = joblib.load(bert_dir / "classifier.joblib")
print("Loaded all three saved models. ParsBERT device:", bert_device)


f:\Roham\projects\Web\Django\vazheh-kav\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded all three saved models. ParsBERT device: cuda


In [4]:
def fasttext_line(text):
    # fastText expects one sample per line; collapse embedded whitespace.
    return " ".join(str(text).split())


def predict_fasttext(model, texts):
    lines = [fasttext_line(text) for text in texts]
    predicted_labels, _ = model.predict(lines, k=1)
    return [LABELS[item[0].removeprefix("__label__")]
            for item in predicted_labels]

@torch.inference_mode()
def encode_comments_bert(texts):
    vectors = []
    batch_size = 16 if bert_device.type == "cuda" else 8
    values = texts.tolist()
    for start in range(0, len(values), batch_size):
        batch = values[start:start + batch_size]
        tokens = bert_tokenizer(
            batch, padding=True, truncation=True,
            max_length=bert_manifest["max_length"], return_tensors="pt",
        )
        tokens = {name: value.to(bert_device) for name, value in tokens.items()}
        token_vectors = bert_encoder(**tokens).last_hidden_state
        mask = tokens["attention_mask"].unsqueeze(-1)
        pooled = (token_vectors * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        vectors.append(pooled.cpu().numpy())
    return np.vstack(vectors)

predictions = {
    "TF-IDF": tfidf_model.predict(test["processed_body"]),
    "fastText": predict_fasttext(fasttext_model, X_test_light),
    "Frozen ParsBERT": bert_classifier.predict(encode_comments_bert(X_test_light)),
}


## Compare on the shared test set


In [5]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

comparison = pd.DataFrame([
    {
        "approach": approach,
        "selected_model": metrics[approach]["selected_model"],
        "cv_macro_f1": metrics[approach]["cv_macro_f1"],
        "test_macro_f1": f1_score(y_test, predictions[approach], average="macro"),
        "test_accuracy": accuracy_score(y_test, predictions[approach]),
    }
    for approach in artifact_paths
]).sort_values("cv_macro_f1", ascending=False, kind="stable").reset_index(drop=True)
display(comparison)
for approach, predicted in predictions.items():
    print(approach)
    print(classification_report(
        y_test, predicted, labels=list(LABEL_NAMES),
        target_names=[LABEL_NAMES[label] for label in LABEL_NAMES],
        digits=3, zero_division=0,
    ))
print("Test scores are exploratory and were not used for model selection.")


,approach,selected_model,cv_macro_f1,test_macro_f1,test_accuracy
0,TF-IDF,LinearSVC,0.892787,0.906453,0.906453
1,fastText,fastText supervised,0.882014,0.899939,0.899950
2,Frozen ParsBERT,SVC,0.876123,0.895932,0.895948


TF-IDF
                 precision    recall  f1-score   support

not_recommended      0.905     0.908     0.907      1000
    recommended      0.908     0.905     0.906       999

       accuracy                          0.906      1999
      macro avg      0.906     0.906     0.906      1999
   weighted avg      0.906     0.906     0.906      1999

fastText
                 precision    recall  f1-score   support

not_recommended      0.892     0.910     0.901      1000
    recommended      0.908     0.890     0.899       999

       accuracy                          0.900      1999
      macro avg      0.900     0.900     0.900      1999
   weighted avg      0.900     0.900     0.900      1999

Frozen ParsBERT
                 precision    recall  f1-score   support

not_recommended      0.887     0.908     0.897      1000
    recommended      0.906     0.884     0.895       999

       accuracy                          0.896      1999
      macro avg      0.896     0.896     0.896  

## Save the chosen model

TF-IDF has the highest training CV macro-F1 (0.892787) and the highest observed test macro-F1 (0.906453). Copy its calibrated pipeline to the path used by Django.


In [6]:
from shutil import copy2

source_model = PROJECT_ROOT / "notebooks" / "recommendation detection" / "output models" / "tfidf_pipeline.joblib"
final_model = PROJECT_ROOT / "web" / "services" / "models" / "recommendation_detection_pipeline.joblib"
final_model.parent.mkdir(parents=True, exist_ok=True)
copy2(source_model, final_model)
print(f"Copied calibrated TF-IDF model to {final_model}")


Copied calibrated TF-IDF model to f:\Roham\projects\Web\Django\vazheh-kav\web\services\models\recommendation_detection_pipeline.joblib
